**Cell #01**

# RAG11 Nutrition — Stage 1.1: Extract & Chunk

Lists the source PDFs directly from the public Google Drive folder (no hardcoded file ids), pulls any that aren't
already sitting locally, extracts hierarchical sections using the per-source algorithm in
`./stage1_1_eda_packages/` (one Python module per source PDF, matched by filename; unknown files fall back to a
generic chunker), and writes parent/child chunk JSON files.

- Input PDFs land in `./stage1_eda_input/sourceN/`
- Chunk output lands in `./stage1_eda_output/sourceN/`
- A `rag11_data_sources`-ready row per source lands in `./stage1_eda_output/sources/` (loaded into Supabase by
  `stage1_2_eda_load_chunks.ipynb`)

**All the code lives in `reusable_code/stage1/extract_chunk.py`**; this notebook just calls its steps one by one so you
can inspect each result. To run the whole pipeline (1.1 → 1.2 → 1.9) without notebooks: `./run_stage1_all.command`.

Run cells top to bottom. The Drive listing, downloads and page-text extraction are cached/idempotent, so re-running
after tuning a section-detection module does not re-download or re-parse anything already on disk.

In [ ]:
# Cell #02
%pip install -q -r requirements.txt

**Cell #03**

## Config

Read from `.env` (restart the kernel after editing it):

- `MAX_NUMBER_OF_PAGES_TO_USE=100` — cap on the pages of *text* extracted per PDF, a fast smoke test (also the default
  when the line is missing). `NONE` = no cap, the full real run. Section *boundaries* still come from the whole PDF; a
  section that starts past the cap just has empty text, so this is NOT a substitute for a full run before trusting output.
- `GOOGLE_DRIVE_SOURCES_FOLDER` — the public "anyone with the link" Drive folder (defaults to the project's folder).

In [ ]:
# Cell #04
from reusable_code.stage1 import extract_chunk as s11

cfg = s11.load_config()

**Cell #05**

## Sources — the Drive folder listing

Known books keep their historical `sourceN` slot (order of `KNOWN_SOURCE_EDA_META`); new files are appended by name;
files a module marks as unusable are skipped before any slot number is assigned.

In [ ]:
# Cell #06
SOURCES = s11.build_sources(cfg)

**Cell #07**

## Download (only if missing locally)

Idempotent: skips any PDF already in `stage1_eda_input/<source>/`. Uses `gdown`, which handles Drive's large-file
confirmation flow, and downloads in parallel (max 6 at a time). Each PDF's page count is checked against the expected one.

In [ ]:
# Cell #08
downloaded_paths, source_fetch_info = s11.download_sources(cfg, SOURCES)

**Cell #09**

## Write source rows — ready for `rag11_data_sources`

One row per source, in the same `rowGUID` / `rowOwnerGUID` / `rowParentGUID` / `orderInList` / `rowJSON` shape as the
chunk tables. A source is the root of its own tree, so it owns itself (`rowOwnerGUID == rowGUID`) and has no parent.

In [ ]:
# Cell #10
s11.write_source_manifests(cfg, SOURCES, source_fetch_info)

**Cell #11**

## Extract raw per-page text (cached)

One entry per real page (index 0 = page 1); only the first `MAX_NUMBER_OF_PAGES_TO_USE` pages actually go through
`get_text()`. NUL characters are stripped here, because Postgres cannot store them.

In [ ]:
# Cell #12
pages_by_source = s11.extract_all_pages(cfg, SOURCES, downloaded_paths)

**Cell #13**

## Section detection (per source)

Each book's own algorithm (native outline, TOC/regex, one section per sūtra, ...) lives in `./stage1_1_eda_packages/`
as one module per PDF, matched by filename. A file with no module is chunked by `generic_fallback.py` (PDF outline →
larger-font headings → fixed page windows) and the strategy used is recorded in its manifest row.

In [ ]:
# Cell #14
sections_by_source = s11.assemble_sections(cfg, SOURCES, downloaded_paths, pages_by_source, source_fetch_info)

**Cell #15**

## Chunking — parent (full section) + child (~400 tokens, 12.5% overlap)

Child boundaries are computed on token counts (`tiktoken`, `cl100k_base`), and each child is prefixed with a short
contextual header (`[Source: ... | Section: ... | Pages N-M]`, 1-based pages) before embedding. A source module can opt
out of the fixed-token split by giving a section its own pre-split `"children"` (named subsections, tables, one
sūtra...): see `child_pieces_for_section()`.

Output: `parent_chunk-N.json` / `child_chunk-parentN-chunkM.json`. Stale files from a previous run are deleted first.

In [ ]:
# Cell #16
chunk_counts = s11.write_all_chunks(cfg, SOURCES, sections_by_source)

**Cell #17**

## Next steps

- **Tuning a source**: its detection logic and `EXPECTED_PAGES`/`STRUCTURE` live in `./stage1_1_eda_packages/sourceN_<slug>.py`;
  tune and re-run this notebook, nothing here needs to change.
- **New sources**: drop the PDF into the Drive folder. With no module it is chunked by the generic fallback; write a
  `sourceN_<slug>.py` module (`FILENAME`, `EXPECTED_PAGES`, `STRUCTURE`, `extract_sections(...)`) for better boundaries.
- Then run `stage1_2_eda_load_chunks.ipynb` (or `./run_stage1_all.command --from 1.2`).

In [ ]:
%%sql
SELECT
    rowJSON.source_key AS source,
    rowJSON.filename AS filename,
    rowJSON.actual_page_count AS pages
FROM read_json_auto('stage1_eda_output/sources/*.json')
ORDER BY orderInList
-- Cell #18